# TRADING INTELLIGENT · CH05 · LABORATOIRE 01
## De la régression classique au Machine Learning supervisé

**Questions directrices.** Comment apprend-on une droite, une relation multivariée, une probabilité et une règle non linéaire ? Que fait exactement `fit()` ? Comment reconnaître le surapprentissage ?

**Pré-requis :** manuel CH05 §§5.1–5.5, 5.10–5.12 ; Python élémentaire. **Autonomie :** ce notebook s'exécute intégralement dans Colab ou en local, sans internet, sans téléchargement de CSV. Les données sont **fictives**.

**Mode d'emploi :** lire la question avant le code ; prédire mentalement le résultat ; exécuter ; interpréter. Les cellules « À vous » sont des espaces de recherche qui ne bloquent pas l'exécution. Les solutions détaillées sont réservées au MASTER. Les graphiques ne sont pas des résultats de trading.

**Convention :** les premiers rendements sont en **points de pourcentage** (ex. `0.4` signifie +0,4 %). D'autres expériences utiliseront une variable sans unité ; chaque changement sera annoncé.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.linear_model import LinearRegression, Ridge, Lasso, LogisticRegression
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier, plot_tree
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.metrics import (mean_absolute_error, mean_squared_error, accuracy_score,
    confusion_matrix, precision_score, recall_score, f1_score, log_loss, brier_score_loss)
from pathlib import Path
SEED=20261116
PETROL='#174B5B'; GOLD='#BA8C4C'; INK='#23343B'; BLUE='#418D9B'
plt.rcParams.update({'figure.figsize':(9,4),'axes.grid':True,'grid.alpha':.16,'axes.spines.top':False,
                     'axes.spines.right':False,'font.size':10})
print('Environnement disponible | numpy',np.__version__,'| pandas',pd.__version__)

## A · Observer avant de calculer

**Pourquoi cette étape ?** Une régression doit d’abord répondre à une question compréhensible.

*Référence du manuel : 5.1–5.2.*

Nous reprenons **exactement** le petit exercice non temporel du manuel 5.2 : les couples $(-1,-1),(0,1),(1,2)$. **Avant de tracer :** la droite idéale passe-t-elle par tous les points ? Quels éléments restent inexpliqués ?

In [ ]:
x=np.array([-1.,0.,1.]); y=np.array([-1.,1.,2.]); n=len(x)
assert n==3
fig,ax=plt.subplots();ax.scatter(x,y,s=75,color=PETROL);ax.set(xlabel='x (variable explicative)',ylabel='y (cible)',title='Trois observations : que peut-on apprendre ?');plt.show()
print(pd.DataFrame({'x':x,'y':y}).to_string(index=False))

**À vous — Lecture du nuage.** Proposez une pente positive ou négative et expliquez la différence entre un point et une droite estimée.

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## B · Construire OLS à la main

**Pourquoi cette étape ?** Nous savons ce que représente une erreur verticale ; apprenons les coefficients avant la bibliothèque Python.

*Référence du manuel : 5.2.*

**Pause calcul (sur papier).** Calculez $\bar x$, $\bar y$, $S_{xy}$ et $S_{xx}$, puis $\hat\beta_1=S_{xy}/S_{xx}$ et $\hat\beta_0=\bar y-\hat\beta_1\bar x$. Prédisez $\hat y$ et le vecteur des résidus $e=y-\hat y$.

In [ ]:
xb=x.mean();yb=y.mean();sxx=np.sum((x-xb)**2);sxy=np.sum((x-xb)*(y-yb))
b1=sxy/sxx;b0=yb-b1*xb
pred=b0+b1*x;res=y-pred;SSE=np.sum(res**2)
print(f'xbar={xb:.3f}, ybar={yb:.3f}, Sxy={sxy:.3f}, Sxx={sxx:.3f}')
print(f'OLS: intercept={b0:.6f}, pente={b1:.6f}, SSE={SSE:.6f}')
display(pd.DataFrame({'x':x,'y':y,'prédit':pred,'résidu':res,'résidu²':res**2}).round(5))
assert np.isclose(b1,1.5) and np.isclose(b0,2/3)
assert np.isclose(np.sum(res),0) and np.isclose(np.dot(x,res),0)

In [ ]:
xx=np.linspace(-1.3,1.3,120);fig,ax=plt.subplots()
ax.scatter(x,y,color=PETROL,s=65,label='Observations');ax.plot(xx,b0+b1*xx,color=BLUE,label='Droite OLS')
for i,(xi,yi) in enumerate(zip(x,y)):
    ax.vlines(xi,b0+b1*xi,yi,colors=GOLD,linestyles='--',linewidth=2,label='Résidu' if i==0 else None)
ax.set(xlabel='x',ylabel='y',title='Les résidus sont des distances verticales signées');ax.legend();plt.show()

**À vous — Calcul manuel.** Retrouvez les trois résidus et leur somme. Pourquoi l’orthogonalité des résidus à $x$ ne prouve-t-elle pas le pouvoir prédictif ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## B bis · Apprendre avec le gradient, un pas à la fois

**Pourquoi cette étape ?** Les moindres carrés ont une solution exacte ; une optimisation numérique donne une deuxième manière de comprendre le mot apprendre.

*Référence du manuel : 5.5.*

Avec la matrice augmentée $\widetilde X=[1,x]$ et $J(\theta)=\frac{1}{2n}\|\widetilde X\theta-y\|^2$, le gradient vaut $\nabla J=\widetilde X^\top(\widetilde X\theta-y)/n$. Partons de $\theta=(0,0)$. **Prédisez :** le premier pas augmente-t-il ou diminue-t-il la perte ?

In [ ]:
D=np.column_stack([np.ones(len(x)),x])
theta_g=np.zeros(2);eta=.2
history=[]
for step in range(35):
    residual_g=D@theta_g-y
    cost=.5*np.mean(residual_g**2)
    history.append([step,theta_g[0],theta_g[1],cost])
    gradient=D.T@residual_g/len(x)
    theta_g=theta_g-eta*gradient
H=pd.DataFrame(history,columns=['iteration','intercept','pente','J'])
display(H.iloc[[0,1,2,5,15,34]].round(5))
assert np.all(np.diff(H['J'])<0)
fig,ax=plt.subplots();ax.plot(H.iteration,H.J,marker='o',markersize=3,color=PETROL)
ax.axhline(SSE/(2*len(x)),ls='--',color=GOLD,label='minimum analytique')
ax.set(xlabel='Itération',ylabel='Perte J',title='Apprendre = faire descendre une fonction de perte');ax.legend();plt.show()
print('Coefficient après 35 pas :',theta_g,'| cible OLS :',[b0,b1])

**Pourquoi ne pas employer toujours le gradient ?** Sur cette petite régression simple, la formule OLS est directe. Sur d'autres familles de modèles et de grandes données, on utilise souvent des optimiseurs itératifs. Le schéma reste commun : paramètres → prédictions → perte → variation des paramètres.

**À vous — Un pas de gradient.** Sur papier, calculez $J(0,0)$ puis le premier gradient. Que fait le taux d’apprentissage ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## C · Ce que fait réellement `.fit()`

**Pourquoi cette étape ?** On demande désormais à Python de retrouver les mêmes coefficients, puis on sépare estimation et prédiction.

*Référence du manuel : 5.2 et 5.5.*

In [ ]:
reg=LinearRegression().fit(x.reshape(-1,1),y)
print('coefficient:',reg.coef_[0],'intercept:',reg.intercept_)
assert np.allclose([reg.intercept_,reg.coef_[0]],[b0,b1])
new_x=np.array([2.,-2.]).reshape(-1,1)
print('Prévisions pour deux nouvelles entrées (sans cible observée) :',reg.predict(new_x))

**Interprétation.** `fit(X,y)` **estime** les paramètres à partir des couples connus. `predict(X_new)` applique les paramètres **déjà appris** à de nouvelles entrées. Une prédiction n’est pas une observation. Un résultat pour $x=2$ est ici une extrapolation, pas une vérité mesurée.

## D · Revenir aux six observations financières du manuel

**Pourquoi cette étape ?** Le jeu simple servira à distinguer faible erreur passée et véritable évaluation ultérieure.

*Référence du manuel : 5.1–5.2.*

In [ ]:
x6=np.array([-2,-1,0,1,2,-1.5],float)
y6=np.array([-.6,-.2,0,.4,.6,-.5],float)
model6=LinearRegression().fit(x6.reshape(-1,1),y6)
assert np.isclose(model6.coef_[0],3.675/11.875)
print(f'β₀={model6.intercept_:.5f} ; β₁={model6.coef_[0]:.5f} ; unités: points de %')
fig,ax=plt.subplots();ax.scatter(x6,y6,color=PETROL);xgrid=np.linspace(-2.3,2.3,100)
ax.plot(xgrid,model6.predict(xgrid.reshape(-1,1)),color=BLUE,label='OLS sur six points')
ax.plot(xgrid,.30*xgrid,color=GOLD,ls='--',label='Règle initiale 0,30x');ax.legend();ax.set(xlabel='Marché (pt %)',ylabel='Actif (pt %)',title='Pourquoi ne pas choisir la pente au hasard ?');plt.show()

**À vous — Sens économique.** Que signifie $\hat\beta_1≈0,3095$ si $x$ augmente de 1 point de pourcentage ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## D bis · Évaluer la même droite sur deux observations réellement réservées

**Pourquoi cette étape ?** Nous retrouvons exactement les dates 7 et 8 des sections 5.1 et 5.2 : elles n’ont pas servi à ajuster la droite.

*Référence du manuel : 5.1–5.2.*

In [ ]:
x_future=np.array([1.5,-1.0]);y_future=np.array([-.2,-.1]) # points de %
p_ols=model6.predict(x_future.reshape(-1,1))
p_regle=.30*x_future
comparaison=pd.DataFrame({'vérité':y_future,'prévision zéro':np.zeros(2),
                            'prévision règle 0.30x':p_regle,'prévision OLS':p_ols})
display(comparaison.round(4))
for name,pred_ in [('Zéro',np.zeros(2)),('Règle 0,30x',p_regle),('OLS',p_ols)]:
    print(name,'MAE test en points de % =',round(mean_absolute_error(y_future,pred_),5))
assert np.isclose(mean_absolute_error(y_future,p_ols),.43684210526315786)

**Diagnostic.** La prévision nulle, bien que plus fruste, est meilleure sur ces **deux** dates de test. Cet échantillon est volontairement microscopique : il illustre la logique de généralisation sans permettre d'affirmer une supériorité stable d'une stratégie.

## E · La régression multiple et les matrices

**Pourquoi cette étape ?** Quand une seule information ne suffit plus, ajoutons une deuxième variable en comprenant chaque dimension.

*Référence du manuel : 5.3.*

In [ ]:
Xm=np.array([[-1,-1],[-1,1],[1,-1],[1,1]],float)
ym=np.array([-.33,-.07,.03,.37])
Xaug=np.c_[np.ones(len(Xm)),Xm]
theta=np.linalg.lstsq(Xaug,ym,rcond=None)[0]
print('X:',Xm.shape,'| X augmentée:',Xaug.shape,'| y:',ym.shape,'| [β0,β1,β2]:',np.round(theta,3))
assert Xm.shape==(4,2) and Xaug.shape==(4,3)
reg_m=LinearRegression().fit(Xm,ym)
assert np.allclose(np.r_[reg_m.intercept_,reg_m.coef_],theta)
display(pd.DataFrame({'signal_marché':Xm[:,0],'signal_secondaire':Xm[:,1],'y':ym,'ŷ':reg_m.predict(Xm)}))

In [ ]:
from mpl_toolkits.mplot3d import Axes3D
fig=plt.figure(figsize=(9,5));ax=fig.add_subplot(111,projection='3d')
g1,g2=np.meshgrid(np.linspace(-1.2,1.2,12),np.linspace(-1.2,1.2,12))
ax.plot_surface(g1,g2,theta[0]+theta[1]*g1+theta[2]*g2,alpha=.27,color=BLUE)
ax.scatter(Xm[:,0],Xm[:,1],ym,s=60,color=GOLD)
ax.set(xlabel='x₁',ylabel='x₂',zlabel='y',title='Une droite devient un plan');plt.show()

**À vous — Dimensions.** Pourquoi ajoute-t-on une colonne de 1 ? Et que devient $X$ avec 50 dates et 6 caractéristiques ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## F · Colinéarité : informations ou répétitions ?

**Pourquoi cette étape ?** Deux variables presque identiques peuvent rendre les coefficients instables sans altérer beaucoup les prédictions.

*Référence du manuel : 5.3.*

In [ ]:
rng=np.random.default_rng(SEED);x1=rng.normal(size=70)
x2=x1+0.002*rng.normal(size=70)
yc=1.0+2*x1+rng.normal(0,.14,size=70)
XC=np.column_stack([x1,x2]);modC=LinearRegression().fit(XC,yc)
print('Corr(x1,x2)=',np.corrcoef(x1,x2)[0,1],'| condition:',np.linalg.cond(XC))
print('Coefficients instables possibles:',np.round(modC.coef_,2),'| somme:',round(modC.coef_.sum(),2))
fig,ax=plt.subplots();ax.scatter(x1,x2,color=PETROL,alpha=.7);ax.set(xlabel='x₁',ylabel='x₂',title='Deux colonnes presque redondantes');plt.show()

**À vous — Colinéarité.** Remplacez l’écart 0,002 par 0,2 puis comparez le conditionnement. Pourquoi le signe d’un coefficient peut-il être peu robuste ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## G · Surapprentissage : voir et mesurer

**Pourquoi cette étape ?** Nous ajustons des polynômes de degrés différents sur des observations bruitées et réservons une validation distincte.

*Référence du manuel : 5.11.*

In [ ]:
rng=np.random.default_rng(47)
xtr=np.sort(rng.uniform(-2,2,22));xva=np.linspace(-2,2,120)
truth=lambda z: .35*z-.2*z**3+.25*np.sin(2*z)
ytr=truth(xtr)+rng.normal(0,.13,len(xtr));yva=truth(xva)+rng.normal(0,.13,len(xva))
poly_res=[];fig,ax=plt.subplots();ax.scatter(xtr,ytr,s=28,color=INK,label='Entraînement')
for deg,cl in [(1,GOLD),(3,BLUE),(14,'#9d4c70')]:
    model=make_pipeline(PolynomialFeatures(deg,include_bias=False),StandardScaler(),Ridge(alpha=1e-4))
    model.fit(xtr.reshape(-1,1),ytr)
    a=mean_squared_error(ytr,model.predict(xtr.reshape(-1,1)))**.5
    b=mean_squared_error(yva,model.predict(xva.reshape(-1,1)))**.5
    poly_res.append((deg,a,b))
    ax.plot(xva,model.predict(xva.reshape(-1,1)),color=cl,label=f'degré {deg} (val={b:.3f})')
ax.set(xlabel='x',ylabel='y',title='Courbes trop rigides ou trop flexibles ?',ylim=(-2,2));ax.legend();plt.show()
print(pd.DataFrame(poly_res,columns=['degré','RMSE train','RMSE validation']).round(3))

**À vous — Sous/surapprentissage.** Pourquoi la plus petite erreur de formation ne constitue-t-elle pas une règle de sélection ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## H · Ridge et Lasso : contraindre une solution

**Pourquoi cette étape ?** Nous pénalisons les grands coefficients sur des variables standardisées.

*Référence du manuel : 5.10–5.11.*

In [ ]:
rng=np.random.default_rng(SEED+1);XX=rng.normal(size=(200,6))
XX[:,1]=XX[:,0]+0.14*rng.normal(size=200)
yR=.8*XX[:,0]-.45*XX[:,3]+rng.normal(0,.55,size=200)
cut=140;modelsR={'OLS':make_pipeline(StandardScaler(),LinearRegression()),
  'Ridge α=20':make_pipeline(StandardScaler(),Ridge(alpha=20)),
  'Lasso α=0,08':make_pipeline(StandardScaler(),Lasso(alpha=.08,max_iter=30000))}
for name,mo in modelsR.items():
    mo.fit(XX[:cut],yR[:cut]);print(name,'| MAE validation=',round(mean_absolute_error(yR[cut:],mo.predict(XX[cut:])),3),
        '| coef=',np.round(mo[-1].coef_,2))

In [ ]:
fig,ax=plt.subplots(figsize=(9,3.6))
for name,mo in modelsR.items():ax.plot(range(1,7),mo[-1].coef_,'o-',label=name)
ax.axhline(0,color=INK,lw=.7);ax.set(xlabel='Variable explicative',ylabel='Coefficient standardisé',title='La pénalité modifie le partage des coefficients');ax.legend();plt.show()

**À vous — Régularisation.** Expliquez pourquoi Lasso peut mettre un coefficient exactement à zéro et Ridge habituellement non.

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## I · Classification : passer des valeurs aux événements

**Pourquoi cette étape ?** Une hausse devient une cible binaire. Une sigmoïde transforme un score réel en probabilité.

*Référence du manuel : 5.4.*

In [ ]:
z=np.linspace(-6,6,300);sigmoid=lambda x:1/(1+np.exp(-x))
fig,ax=plt.subplots();ax.plot(z,sigmoid(z),color=PETROL,lw=2)
for threshold in [.3,.5,.7]:ax.axhline(threshold,ls='--',alpha=.45,label=f'seuil {threshold}')
ax.set(xlabel='Score z',ylabel='σ(z)',title='Du score linéaire à la probabilité',ylim=(-.03,1.03));ax.legend();plt.show()
for zz in [-2,0,2]:print('score',zz,'→ p=',round(float(sigmoid(zz)),4))

In [ ]:
rng=np.random.default_rng(SEED+2);Xc=rng.normal(size=(400,2))
latent=-.3+1.4*Xc[:,0]-.65*Xc[:,1];ptrue=sigmoid(latent)
labels=(rng.random(400)<ptrue).astype(int)
clf=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000))
clf.fit(Xc[:260],labels[:260]);probs=clf.predict_proba(Xc[260:])[:,1]
print('Dimensions:',Xc.shape,'| premières probabilités:',np.round(probs[:6],3))
print('Somme probas deux classes=',clf.predict_proba(Xc[260:])[:3].sum(axis=1))

## J · Le seuil n’est pas l’apprentissage

**Pourquoi cette étape ?** Changer le seuil modifie la décision sans réentraîner la régression logistique.

*Référence du manuel : 5.4 et 5.12.*

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
rows=[]
for th in [.30,.50,.70]:
    yh=(probs>=th).astype(int);rows.append([th,accuracy_score(labels[260:],yh),precision_score(labels[260:],yh,zero_division=0),
    recall_score(labels[260:],yh,zero_division=0),f1_score(labels[260:],yh,zero_division=0)])
display(pd.DataFrame(rows,columns=['seuil','accuracy','précision','rappel','F1']).round(3))
fig,axes=plt.subplots(1,3,figsize=(11,3.3))
for ax,th in zip(axes,[.30,.50,.70]):
    mat=confusion_matrix(labels[260:],probs>=th,labels=[0,1])
    ConfusionMatrixDisplay(mat,display_labels=['Non-hausse','Hausse']).plot(ax=ax,cmap='Blues',colorbar=False)
    ax.set_title(f'Seuil {th:.2f}');ax.grid(False)
plt.tight_layout();plt.show()

**À vous — Seuil de décision.** Comment le rappel évolue-t-il lorsque le seuil augmente ? Pourquoi la classe positive doit-elle être définie explicitement ?

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

## K · Accuracy trompeuse et calibration

**Pourquoi cette étape ?** Un modèle qui devine toujours la majorité peut avoir une forte accuracy ; un seuil ne mesure pas la qualité des probabilités.

*Référence du manuel : 5.4, 5.12.*

In [ ]:
yimbal=np.r_[np.zeros(90,dtype=int),np.ones(10,dtype=int)]
major=np.zeros(100,dtype=int);print('Toujours classe majoritaire : accuracy',accuracy_score(yimbal,major),'rappel hausses',recall_score(yimbal,major))
print('Log-loss logistique :',round(log_loss(labels[260:],probs,labels=[0,1]),4),
      '| Brier :',round(brier_score_loss(labels[260:],probs),4))
cal=pd.DataFrame({'p':probs,'y':labels[260:]})
cal['bin']=pd.cut(cal['p'],bins=np.linspace(0,1,7),include_lowest=True)
g=cal.groupby('bin',observed=True).agg(pred=('p','mean'),frequence=('y','mean'),n=('y','size'))
display(g.round(3))
fig,ax=plt.subplots();ax.plot([0,1],[0,1],ls='--',color=INK,label='Calibration idéale');ax.scatter(g.pred,g.frequence,s=20+3*g.n,color=GOLD,label='Échantillon de contrôle');ax.set(xlabel='Probabilité moyenne annoncée',ylabel='Fréquence observée',title='Une probabilité exige une vérification empirique',xlim=(0,1),ylim=(0,1));ax.legend();plt.show()

## L · Un arbre découvre une autre forme de règle

**Pourquoi cette étape ?** Une partition non linéaire doit elle aussi se justifier hors échantillon.

*Référence du manuel : 5.10–5.11.*

In [ ]:
tree2=DecisionTreeClassifier(max_depth=2,min_samples_leaf=15,random_state=SEED)
tree8=DecisionTreeClassifier(max_depth=8,min_samples_leaf=2,random_state=SEED)
for model in [tree2,tree8]:
    model.fit(Xc[:260],labels[:260]); print('profondeur',model.get_depth(),'accuracy train',round(model.score(Xc[:260],labels[:260]),3),'validation',round(model.score(Xc[260:],labels[260:]),3))
fig,ax=plt.subplots(figsize=(11,5));plot_tree(tree2,feature_names=['x1','x2'],class_names=['non-hausse','hausse'],filled=True,rounded=True,ax=ax,fontsize=9);plt.show()

## M · Synthèse : une méthode, plusieurs sorties

**Pourquoi cette étape ?** L’architecture d’apprentissage est commune, la cible et les métriques ne le sont pas.

*Référence du manuel : 5.5 et 5.12.*

**Carte à retenir.** Données et cible → type de problème → fonction de perte → estimation des paramètres → prévision → comparaison sur données non utilisées pour l’ajustement. Pour une régression numérique : MAE/RMSE ; pour une probabilité binaire : log-loss/Brier ; pour une classe : matrice de confusion et métriques dépendant du seuil. **Aucune de ces métriques n’est une rentabilité nette.**

In [ ]:
assert len(probs)==len(labels[260:])
assert np.all((probs>=0)&(probs<=1))
assert np.allclose(clf.predict_proba(Xc[260:]).sum(axis=1),1)
print('Contrôles NB01 : OK — dimensions, probabilités, OLS manuel, algorithmes reproductibles.')

**À vous — Défi intégrateur.** Rédigez un protocole en 6 lignes pour comparer OLS, Ridge et une logistique, sans confondre cibles et métriques.

In [ ]:
# Votre espace de travail (facultatif pour exécuter la suite)
# Écrivez ici vos essais, puis confrontez-les aux résultats précédents.

### Checklist de maîtrise — NB01
- [ ] Je peux calculer et vérifier une droite OLS à la main.
- [ ] Je sais distinguer lignes, colonnes, coefficients et intercept.
- [ ] Je peux expliquer la colinéarité, Ridge et Lasso.
- [ ] Je distingue une sortie réelle, une probabilité et une classe.
- [ ] Je sais expliquer le rôle du seuil, de la log-loss et de la calibration.
- [ ] Je sais pourquoi la meilleure erreur d’entraînement ne désigne pas le meilleur modèle.

**Prolongement NB02.** Nous allons maintenant donner aux observations un ordre temporel : AR(1) devient régression, les retards deviennent colonnes, et chaque cible future exige un calendrier rigoureux.